#Lab: Workspace & Catalog
 
##**Course 1, Week 1: Lakehouse Architecture & Platform**

###Objectives
- Navigate the Databricks Workspace
- Explore the Unity Catalog hierarchy (Metastore > Catalog > Schema > Table)
- Use DBFS to browse files
- Inspect compute resources



##Part 1: Catalog Exploration
###List all available catalogs and schemas.

In [0]:
# List all catalogs
spark.sql("SHOW CATALOGS").display()

# Get all catalogs
catalogs = spark.sql("SHOW CATALOGS").collect()

# Show schemas for each catalog
for catalog in catalogs:
    catalog_name = catalog.catalog
    print(f"\n📁 Catalog: {catalog_name}")
    spark.sql(f"SHOW SCHEMAS IN {catalog_name}").display()

##Part 2: Create Your Own Schema
###Create a schema and table in the catalog.

In [0]:
# Create schema in the current catalog
spark.sql("CREATE SCHEMA IF NOT EXISTS lab_workspace")

# Create cities table
spark.sql("""
  CREATE TABLE IF NOT EXISTS lab_workspace.cities (
    name STRING,
    state STRING,
    population BIGINT
  )
""")

# Insert sample cities
spark.sql("""
  INSERT INTO lab_workspace.cities VALUES
    ('San Francisco', 'California', 873965),
    ('Austin', 'Texas', 961855),
    ('Seattle', 'Washington', 749256),
    ('Denver', 'Colorado', 715522)
""")

# Query the table
spark.sql("SELECT * FROM lab_workspace.cities ORDER BY population DESC").display()


##Part 3: File System Exploration
###Explore DBFS and Databricks sample datasets

In [0]:
# List contents of databricks-datasets
display(dbutils.fs.ls("/databricks-datasets/"))

# Preview a sample file
print(dbutils.fs.head("/databricks-datasets/nyctaxi/readme_nyctaxi.txt"))

##Part 4: Compute Information
###Inspect your current cluster.

In [0]:
# Get Spark version
print(f"Spark Version: {spark.version}")

# Helper function to safely get config
def get_config_safe(key, default="Not available (Serverless)"):
    try:
        return spark.conf.get(key)
    except:
        return default

# Try to get configurations (many won't be available on Serverless)
print(f"Driver Memory: {get_config_safe('spark.driver.memory')}")
print(f"Executor Memory: {get_config_safe('spark.executor.memory')}")
print(f"Executor Cores: {get_config_safe('spark.executor.cores')}")

##Lab Validation and Cleanup

In [0]:
def validate_lab():
    """Validate lab completion."""
    checks = []

    # Check 1: Schema exists
    try:
        spark.sql("SHOW TABLES IN lab_workspace")
        checks.append(("Schema created", True))
    except Exception:
        checks.append(("Schema created", False))

    # Check 2: Cities table exists with data
    try:
        df = spark.sql("SELECT * FROM lab_workspace.cities")
        checks.append(("Cities table with data", df.count() >= 3))
    except Exception:
        checks.append(("Cities table with data", False))

    print("Lab Validation Results:")
    print("-" * 40)
    all_passed = True
    for name, passed in checks:
        status = "PASS" if passed else "FAIL"
        print(f"  [{status}] {name}")
        if not passed:
            all_passed = False

    if all_passed:
        print("\nAll checks passed! Lab complete.")
    else:
        print("\nSome checks failed. Review your code above.")

validate_lab()

# Clean up
try:
    spark.sql("DROP SCHEMA IF EXISTS lab_workspace CASCADE")
except Exception:
    pass